In [38]:
import re
import os
import numpy as np
from google import genai
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


In [5]:
def split_text_and_code(document: str) -> list[dict]:
    parts = re.split(r"(```.*?```)", document, flags=re.DOTALL)
    
    blocks = []
    for part in parts:
        if not part.strip():
            continue
        if part.startswith("```") and part.endswith("```"):
            blocks.append({"type": 'code', "content": part.strip()})
        else:
            blocks.append({"type": "text", "content": part.strip()})
        
        pass
    
    return blocks

In [6]:
def chunk_text_block(text: str, max_chunk_size: int = 500) -> list[str]:
    paragraphs = text.split("\n\n")
    chunks = []
    
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        
        if len(para) <= max_chunk_size:
            chunks.append(para)
        else:
            # paragraf kepanjangan, pecah jadi kalimat
            sentences = re.split(r'(?<=[.!?])\s+', para)
            # gabungin kalimat-kalimat itu jadi chunk sampai mendekati max_chunk_size
            current_chunk = ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) <= max_chunk_size:
                    current_chunk += " " + sentence
                else:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence
            if current_chunk:
                chunks.append(current_chunk.strip())
    
    return chunks

In [7]:
def chunk_document(document, max_chuck_size =500):
    blocks = split_text_and_code(document)
    chunks = []
    
    for block in blocks:
        if block['type'] =='text':
            text_chunks = chunk_text_block(block['content'], max_chuck_size)
            for tc in text_chunks:
                chunks.append({'type': 'text', 'content': tc})
        else:
            chunks.append({'type': 'code', 'content': block['content']})
    return chunks

In [5]:
with open("data/02_autograd_concept.md", "r") as f:
    content = f.read()

result = chunk_document(content)

for r in result:
    print(r["type"], "-", len(r["content"]))

text - 20
text - 20
text - 456
text - 308
text - 29
text - 136
code - 88
text - 254
text - 22
text - 251
code - 40
text - 234
text - 35
text - 411
text - 30
text - 206
code - 71
text - 172


In [12]:
with open ("02_autograd_concept.md", "r") as f :
    content = f.read()
    
blocks = split_text_and_code(content)

for b in blocks:
    print(b['type'], "-", len(b['content']), 'karakter')
    print(b['content'][:80])
    print('---')

text - 979 karakter
# Autograd Mechanics

## What is Autograd?

Autograd is the component of PyTorch
---
code - 88 karakter
```python
import torch

x = torch.randn(3, requires_grad=True)
y = x * 2
z = y.s
---
text - 531 karakter
In this example, `x` is a tensor that requires gradients. Any tensor derived fro
---
code - 40 karakter
```python
z.backward()
print(x.grad)
```
---
text - 924 karakter
After this call, the `grad` attribute of `x` contains the gradient of `z` with r
---
code - 71 karakter
```python
with torch.no_grad():
    predictions = model(input_data)
```
---
text - 172 karakter
Disabling gradient tracking in this way reduces memory usage and speeds up compu
---


In [14]:
sample_text = blocks[0]["content"]  # block text pertama tadi
paragraphs = sample_text.split("\n\n")

for p in paragraphs:
    print(len(p), "karakter -", p[:50])
    print("---")

20 karakter - # Autograd Mechanics
---
20 karakter - ## What is Autograd?
---
456 karakter - Autograd is the component of PyTorch responsible f
---
308 karakter - When a tensor is created with gradient tracking en
---
29 karakter - ## Enabling Gradient Tracking
---
136 karakter - A tensor only participates in autograd if it is ex
---


In [8]:
folder = 'data'
all_chunks = []

for filename in os.listdir(folder):
    if filename.endswith('.md'):
        with open(os.path.join(folder, filename), 'r') as f:
            content = f.read()
        doc_chunks = chunk_document(content)
        for c in doc_chunks :
            c['source'] = filename
        all_chunks.extend(doc_chunks)
        
print(f"Total chunks: {len(all_chunks)}")
print(f"Text chunks: {sum(1 for c in all_chunks if c['type']=='text')}")
print(f"Code chunks: {sum(1 for c in all_chunks if c['type']=='code')}")

Total chunks: 65
Text chunks: 57
Code chunks: 8


In [ ]:
for item in all_chunks:
    if item['type'] =='code':
        print(f"[{item['source']}]")
        print(item['content'])

[02_autograd_concept.md]
```python
import torch

x = torch.randn(3, requires_grad=True)
y = x * 2
z = y.sum()
```
[02_autograd_concept.md]
```python
z.backward()
print(x.grad)
```
[02_autograd_concept.md]
```python
with torch.no_grad():
    predictions = model(input_data)
```
[03_nn_linear_api_reference.md]
```python
import torch
import torch.nn as nn

layer = nn.Linear(in_features=20, out_features=10)
input_tensor = torch.randn(32, 20)
output_tensor = layer(input_tensor)
print(output_tensor.shape)
```
[04_quickstart_training_tutorial.md]
```python
import torch
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x
```
[

In [9]:
model = SentenceTransformer('all-MiniLM-l6-v2')

test_sentence = "Autograd computes gradients automatically."
embedding = model.encode(test_sentence)

print(type(embedding))
print(embedding.shape)
print(embedding[:5])  

<class 'numpy.ndarray'>
(384,)
[-0.09371744 -0.06475852 -0.01157485  0.01899678  0.01658196]


In [10]:
def prepare_text_for_embedding(chunk: dict) -> str:
    """
    Siapin teks yang akan di-embed.
    Kalau chunk code, tambahin prefix konteks.
    Content asli TETAP disimpan utuh, cuma teks buat embedding yang beda.
    """
    if chunk["type"] == "code":
        return "Python code example: " + chunk["content"]
    else:
        return chunk["content"]

In [11]:
for item in all_chunks:
    text = prepare_text_for_embedding(item)
    result = model.encode(text)
    item['embedding'] = result
    
print(all_chunks[0].keys())
print(all_chunks[0]['embedding'].shape)
print(all_chunks[0]['type'], "-", all_chunks[0]['content'][:60])
    

dict_keys(['type', 'content', 'source', 'embedding'])
(384,)
text - # Understanding Tensors


In [12]:
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """
    Hitung cosine similarity antara dua vector.
    
    Rumus: (dot product antara vec_a dan vec_b) / (magnitude vec_a * magnitude vec_b)
    
    Hint:
    - dot product: np.dot(a, b)
    - magnitude (panjang vector): np.linalg.norm(v)
    """
    v = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    result = np.dot(vec_a, vec_b) / v
    return result

In [13]:
# ambil embedding dari beberapa chunk buat dites
autograd_chunks = [c for c in all_chunks if "autograd" in c['source'].lower()]
tensor_chunks = [c for c in all_chunks if "tensor" in c['source'].lower()]

emb_1 = autograd_chunks[0]['embedding']  # dari file autograd
emb_2 = autograd_chunks[1]['embedding']  # dari file autograd juga (harusnya mirip topik)
emb_3 = tensor_chunks[0]['embedding']    # dari file tensor (topik beda)

print("Autograd vs Autograd:", cosine_similarity(emb_1, emb_2))
print("Autograd vs Tensor:", cosine_similarity(emb_1, emb_3))

Autograd vs Autograd: 0.8111571
Autograd vs Tensor: 0.30787048


In [14]:
def retrieve(query: str, all_chunks: list[dict], top_k: int = 3) -> list[dict]:
    query_embedding = model.encode(query)
    
    for item in all_chunks:
        item['score'] = cosine_similarity(query_embedding, item['embedding'])
    
    sorted_chunks = sorted(all_chunks, key=lambda x: x['score'], reverse=True)
    return sorted_chunks[:top_k]

In [13]:
results = retrieve("how does gradient computation work?", all_chunks, top_k=3)
for r in results:
    print(round(r['score'], 3), "-", r['source'], "-", r['content'][:80])

0.631 - 02_autograd_concept.md - When a tensor is created with gradient tracking enabled, PyTorch builds a comput
0.623 - 02_autograd_concept.md - Once a scalar output has been computed, calling the backward method triggers the
0.595 - 02_autograd_concept.md - ## Computing Gradients


In [ ]:
client = genai.Client(api_key="YOUR_API_KEY")

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Explain what a tensor is in one sentence."
)

print(response.text)

A tensor is a mathematical object that generalizes scalars, vectors, and matrices to any number of dimensions, typically represented as a multi-dimensional array of numbers.


In [15]:
def build_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    """
    Susun prompt dari query + retrieved chunks.
    """
    context = "\n\n".join([chunk['content'] for chunk in retrieved_chunks])
    
    prompt = f"""Kamu adalah asisten yang menjawab pertanyaan HANYA berdasarkan konteks berikut.
                Jika jawabannya tidak ada di konteks, katakan kamu tidak tahu.

                Konteks:
                {context}

                Pertanyaan: {query}

                Jawaban:"""
    
    return prompt

In [16]:
def rag_answer(query: str, all_chunks: list[dict], top_k: int = 3) -> str:
    """
    Pipeline RAG lengkap: retrieve -> build prompt -> generate answer
    """
    retrieved_chunks = retrieve(query, all_chunks, top_k=top_k)
    prompt = build_prompt(query, retrieved_chunks)
    
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )
    
    return response.text

In [20]:
answer = rag_answer("how does gradient computation work?", all_chunks, top_k=3)
print(answer)

Berdasarkan konteks yang diberikan, komputasi gradien bekerja dengan cara berikut:

1. Ketika sebuah tensor dibuat dengan pelacakan gradien diaktifkan, PyTorch membangun graf komputasi (*computation graph*) di balik layar saat operasi dilakukan. Setiap simpul (node) mencatat informasi yang cukup untuk menghitung gradien melalui aturan rantai (*chain rule*).
2. Setelah output skalar dihitung, memanggil metode `backward` akan memicu komputasi gradien.
3. PyTorch menelusuri graf komputasi secara terbalik (*in reverse*), menerapkan aturan rantai di setiap langkah untuk menghitung seberapa besar kontribusi setiap input terhadap hasil akhir.


In [21]:
# sample corpus sederhana dulu
sample_docs = [
    "torch.nn.Conv2d applies a 2D convolution over an input signal",
    "Use torch.optim.Adam for adaptive learning rate optimization",
    "The forward method defines how data flows through the network",
    "torch.nn.Linear applies a linear transformation to incoming data",
]

# BM25 butuh tokenized input (list of tokens per dokumen), bukan raw string
tokenized_docs = [doc.lower().split() for doc in sample_docs]

bm25 = BM25Okapi(tokenized_docs)

# test query
query = "Conv2d convolution"
tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)
print(scores)

[0.82544777 0.         0.         0.        ]


In [20]:
query2 = "optimizer for gradient descent"
tokenized_query2 = query2.lower().split()
scores2 = bm25.get_scores(tokenized_query2)
print(scores2)

[0. 0. 0. 0.]


In [19]:
# stopword list sederhana (bisa pakai nltk atau library lain nanti)
stopwords = {"the", "a", "an", "is", "are", "for", "to", "of", "in", "on", "and", "or", "with"}

def simple_tokenize(text):
    tokens = text.lower().split()
    return [t for t in tokens if t not in stopwords]

tokenized_docs = [simple_tokenize(doc) for doc in sample_docs]
bm25 = BM25Okapi(tokenized_docs)

tokenized_query2 = simple_tokenize(query2)
scores2 = bm25.get_scores(tokenized_query2)
print(scores2)

[0. 0. 0. 0.]


In [24]:
stopwords = {"the", "a", "an", "is", "are", "for", "to", "of", "in", "on", 
             "and", "or", "with", "this", "that", "it", "as", "be", "by"}

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)  # ganti semua non-alphanumeric jadi spasi
    tokens = text.split()
    return [t for t in tokens if t not in stopwords]

# asumsi: all_chunks = list gabungan semua chunk dari 4 dokumen kamu
# (hasil chunk_document() dipanggil untuk tiap dokumen, lalu di-extend jadi satu list)

tokenized_corpus = [simple_tokenize(chunk['content']) for chunk in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, top_k=5):
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    
    # pasangkan skor dengan chunk aslinya, urutkan descending
    ranked = sorted(
        zip(scores, all_chunks), 
        key=lambda x: x[0], 
        reverse=True
    )
    return ranked[:top_k]

# test
results = bm25_search("what is autograd")
for score, chunk in results:
    print(f"{score:.3f} | {chunk['type']} | {chunk['content'][:80]}")

8.860 | text | ## What is Autograd?
4.836 | text | ## What is a Tensor?
4.023 | text | # Autograd Mechanics
2.750 | text | A tensor only participates in autograd if it is explicitly marked to require gra
1.651 | text | Tensors also carry metadata beyond their raw values. Each tensor has a shape, wh


In [25]:
print(simple_tokenize("## What is a Tensor?"))
print(simple_tokenize("## What is Autograd?"))

['what', 'tensor']
['what', 'autograd']


In [28]:
def is_heading_only(text, max_words=8):
    stripped = text.strip()
    match = re.match(r'^#{1,6}\s+(.*)', stripped)
    if not match:
        return False
    heading_text = match.group(1)  # ambil teks SETELAH tanda '#', jadi '#' gak ikut dihitung
    word_count = len(heading_text.split())
    return word_count <= max_words

def merge_heading_chunks(chunks):
    merged = []
    i = 0
    while i < len(chunks):
        if chunks[i]['type'] == 'text' and is_heading_only(chunks[i]['content']):
            group = [chunks[i]['content']]
            j = i + 1
            # terus tarik chunk berikutnya SELAMA dia masih heading-only juga
            while j < len(chunks) and chunks[j]['type'] == 'text' and is_heading_only(chunks[j]['content']):
                group.append(chunks[j]['content'])
                j += 1
            # setelah stack heading habis, tarik satu chunk konten asli untuk digabung
            if j < len(chunks):
                group.append(chunks[j]['content'])
                merged_type = chunks[j]['type']
                j += 1
            else:
                merged_type = 'text'  # edge case: heading di akhir dokumen, tidak ada konten sesudahnya
            merged.append({'type': merged_type, 'content': '\n\n'.join(group)})
            i = j
        else:
            merged.append(chunks[i])
            i += 1
    return merged

In [29]:
all_chunks_merged = merge_heading_chunks(all_chunks)

print(f"Sebelum merge: {len(all_chunks)} chunks")
print(f"Sesudah merge: {len(all_chunks_merged)} chunks")

# cek beberapa contoh buat verifikasi
for c in all_chunks_merged[:5]:
    print(f"[{c['type']}] {c['content'][:100]}")

Sebelum merge: 65 chunks
Sesudah merge: 41 chunks
[text] # Understanding Tensors

## What is a Tensor?

A tensor is the fundamental data structure used throu
[text] Tensors are similar to NumPy arrays in many respects. Both store data in a multi-dimensional grid an
[text] ## Why Tensors Matter for Deep Learning

Every piece of data that flows through a neural network in 
[text] Tensors also carry metadata beyond their raw values. Each tensor has a shape, which describes the si
[text] ## Creating Tensors

There are several common ways to create a tensor. You can create one directly f


In [30]:
tokenized_corpus_merged = [simple_tokenize(chunk['content']) for chunk in all_chunks_merged]
bm25_merged = BM25Okapi(tokenized_corpus_merged)

def bm25_search_merged(query, top_k=5):
    tokenized_query = simple_tokenize(query)
    scores = bm25_merged.get_scores(tokenized_query)
    ranked = sorted(zip(scores, all_chunks_merged), key=lambda x: x[0], reverse=True)
    return ranked[:top_k]

results = bm25_search_merged("what is autograd")
for score, chunk in results:
    print(f"{score:.3f} | {chunk['type']} | {chunk['content'][:80]}")

5.440 | text | # Autograd Mechanics

## What is Autograd?

Autograd is the component of PyTorch
3.010 | text | ## Enabling Gradient Tracking

A tensor only participates in autograd if it is e
1.880 | text | ## The Computation Graph is Dynamic

One of the distinguishing features of PyTor
1.839 | text | Tensors also carry metadata beyond their raw values. Each tensor has a shape, wh
1.760 | text | # Understanding Tensors

## What is a Tensor?

A tensor is the fundamental data 


In [32]:
for chunk in all_chunks_merged:
    chunk['embedding'] = model.encode(chunk['content'])

In [33]:
contents = [chunk['content'] for chunk in all_chunks_merged]
embeddings = model.encode(contents)

for chunk, emb in zip(all_chunks_merged, embeddings):
    chunk['embedding'] = emb

In [35]:
def normalize_scores(scores):
    min_s, max_s = min(scores), max(scores)
    if max_s == min_s:
        return [0.0 for _ in scores]
    return [(s - min_s) / (max_s - min_s) for s in scores]

def hybrid_search(query: str, all_chunks: list[dict], bm25_index, top_k: int = 5, alpha: float = 0.5):
    query_embedding = model.encode(query)
    dense_scores = [cosine_similarity(query_embedding, item['embedding']) for item in all_chunks]

    tokenized_query = simple_tokenize(query)
    bm25_scores = bm25_index.get_scores(tokenized_query)

    dense_norm = normalize_scores(dense_scores)
    bm25_norm = normalize_scores(bm25_scores)

    for item, d_score, b_score in zip(all_chunks, dense_norm, bm25_norm):
        item['dense_score'] = d_score
        item['bm25_score'] = b_score
        item['hybrid_score'] = alpha * d_score + (1 - alpha) * b_score

    sorted_chunks = sorted(all_chunks, key=lambda x: x['hybrid_score'], reverse=True)
    return sorted_chunks[:top_k]

In [36]:
results = hybrid_search("what is autograd", all_chunks_merged, bm25_merged, top_k=5)
for c in results:
    print(f"hybrid={c['hybrid_score']:.3f} | dense={c['dense_score']:.3f} | bm25={c['bm25_score']:.3f} | {c['content'][:70]}")

hybrid=1.000 | dense=1.000 | bm25=1.000 | # Autograd Mechanics

## What is Autograd?

Autograd is the component 
hybrid=0.647 | dense=0.741 | bm25=0.553 | ## Enabling Gradient Tracking

A tensor only participates in autograd 
hybrid=0.385 | dense=0.424 | bm25=0.346 | ## The Computation Graph is Dynamic

One of the distinguishing feature
hybrid=0.242 | dense=0.147 | bm25=0.338 | Tensors also carry metadata beyond their raw values. Each tensor has a
hybrid=0.213 | dense=0.427 | bm25=0.000 | After this call, the `grad` attribute of `x` contains the gradient of 


In [37]:
results = hybrid_search("what is autograd", all_chunks_merged, bm25_merged, top_k=5, alpha=0.8)
for c in results:
    print(f"hybrid={c['hybrid_score']:.3f} | dense={c['dense_score']:.3f} | bm25={c['bm25_score']:.3f} | {c['content'][:70]}")

hybrid=1.000 | dense=1.000 | bm25=1.000 | # Autograd Mechanics

## What is Autograd?

Autograd is the component 
hybrid=0.703 | dense=0.741 | bm25=0.553 | ## Enabling Gradient Tracking

A tensor only participates in autograd 
hybrid=0.408 | dense=0.424 | bm25=0.346 | ## The Computation Graph is Dynamic

One of the distinguishing feature
hybrid=0.342 | dense=0.427 | bm25=0.000 | After this call, the `grad` attribute of `x` contains the gradient of 
hybrid=0.251 | dense=0.313 | bm25=0.000 | It is important to call `optimizer.zero_grad()` at the start of each i


In [39]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json: 100%|██████████| 794/794 [00:00<?, ?B/s] 
c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
model.safetensors: 100%|██████████| 90.9M/90.9M [00:36<00:00, 2.50

In [40]:
def rerank(query: str, candidates: list[dict], top_k: int = 5):
    # bikin pasangan (query, content) untuk tiap kandidat
    pairs = [[query, chunk['content']] for chunk in candidates]
    
    # cross-encoder langsung kasih skor relevansi per pasangan
    rerank_scores = reranker.predict(pairs)
    
    for chunk, score in zip(candidates, rerank_scores):
        chunk['rerank_score'] = score
    
    sorted_chunks = sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)
    return sorted_chunks[:top_k]

In [41]:
def rag_retrieve(query: str, all_chunks: list[dict], bm25_index, 
                  hybrid_top_k: int = 15, final_top_k: int = 5, alpha: float = 0.5):
    # tahap 1: hybrid search, ambil kandidat lebih banyak dari yang dibutuhkan akhir
    candidates = hybrid_search(query, all_chunks, bm25_index, top_k=hybrid_top_k, alpha=alpha)
    
    # tahap 2: rerank kandidat itu, ambil top_k final
    final_results = rerank(query, candidates, top_k=final_top_k)
    return final_results

In [42]:
rag_retrieve("what is autograd", all_chunks_merged, bm25_merged)

[{'type': 'text',
  'content': '# Autograd Mechanics\n\n## What is Autograd?\n\nAutograd is the component of PyTorch responsible for automatic differentiation. In simple terms, it keeps track of every operation performed on a tensor so that it can later compute the gradient of some output with respect to that tensor. This is the mechanism that makes training neural networks through gradient descent practical, since it removes the need for developers to manually derive and implement gradient formulas for every operation in a model.',
  'embedding': array([-7.34598115e-02, -3.33580635e-02, -3.39925103e-02,  1.33535666e-02,
          3.73425372e-02,  1.63255557e-02,  5.70357870e-03,  3.99915967e-03,
         -2.78493017e-02,  1.36604914e-02, -9.73898638e-03,  6.26193359e-02,
         -8.87688696e-02,  4.66637351e-02, -1.03367388e-01,  1.06963713e-03,
         -6.67670295e-02,  2.97348630e-02, -1.01290122e-01, -1.25824854e-01,
         -4.58868183e-02, -1.67048760e-02, -3.21237557e-02,  4.